# 09. AGENTS

- Введение
    - Что такое агенты и зачем они нужны
    - Архитектура агентов
    - Свойства агентов
    - Примеры агентов
- Простой SE-агент
- MCP

# 1. Введение

## Что такое агенты и зачем они нужны

* LLM $-$ генерирует текст, выполняет инструкции. С точки зрения информации LLM ограничена информацией, содержащейся в промпте и в её весах. Если в процессе генерации возникла потребность в новой информации, то это нельзя испроавить. LLM ограничена количеством токенов, в частности, время на ответ ограничено. Если задача простая, то достаточно запроса в LLM.

* **Агент** $-$  система, которая использует LLM для планирования, сбора информации и выполнения действий по достижению цели. Может запросить или добыть информацию, если недостаточно. Возможно итеративное улучшение результатов своих действий. Если задача сложная, то нужен агент или система агентов.

В SE агенты применяются в следующих задачах:
- Исправление ошибок и багов
- Рефакторинг кода
- Генерация тестов
- Оптимизация производительности

## Архитектура агентов

Базовые компоненты системы:
- **LLM**: ядро агента, отвечающее за понимание запросов, рассуждение и генерацию текста
- **координатор-менеджер**: отвечает за планирование, выполнение и т.д.
- **инструменты**: калькулятор, файловая система, git, браузер и т.д. (модель должна быть **специально обучена или дообучена** для работы с инструментами; это называется **function calling** или **tool calling**)
- **память**: хранение информации о предыдущих взаимодействиях

### Механизм принятия решений

Центральным элементом агента является его механизм принятия решений, реализуемый через **Agent Executor** и основанный на паттерне **ReAct (Reason + Act)** ([ссылка](https://arxiv.org/abs/2210.03629)):

```mermaid
graph TD
    A[Наблюдение] --> B[Обдумывание]
    B --> C{Решение}
    C -->|Использовать инструмент| D[Действие]
    C -->|Задача выполнена| E[Финальный ответ]
    D --> F[Наблюдение результата]
    F --> A
```

1. **Наблюдение (Observation)**: Агент получает входные данные — первоначальный запрос пользователя или результат выполнения предыдущего инструмента.
2. **Обдумывание (Thought/Reasoning)**: LLM анализирует входные данные, текущую цель, доступные инструменты и историю. Модель генерирует внутреннее рассуждение о следующем шаге.
3. **Действие (Action)**: На основе рассуждений LLM либо выбирает инструмент для вызова и определяет входные данные для него, либо решает, что задача выполнена, и генерирует финальный ответ.
4. **Исполнение/Ответ**: Если выбран инструмент, Agent Executor вызывает его и получает результат (новое наблюдение). Если сгенерирован финальный ответ, цикл завершается.


## Свойства агентов

Ключевые характеристики агентов:

- **Автономность**: способность действовать без постоянного вмешательства человека
- **Адаптивность**: возможность адаптироваться к изменяющимся условиям и новым задачам
- **Целеустремленность**: ориентация на достижение конкретных целей
- **Взаимодействие**: способность использовать инструменты и API для выполнения задач

# 2. Простой SE-агент

## Фреймворки для создания агентов

- **Langchain** — опенсорсный фреймворк, предназначенный для упрощения разработки приложений на основе LLM.
- **Qwen-Agent** — специализированный фреймворк для работы с моделями серии Qwen, предоставляющий инструменты для создания агентов с поддержкой вызова функций и работы с инструментами.
- **Hugging Face Transformers Agents** (перемещены в **smolagents**) — предоставляли унифицированный интерфейс для работы с различными моделями и инструментами.


| **Фреймворк** | **Язык** | **Поддержка моделей** | **Инструменты** | **Сложность** |
|---------------|----------|----------------------|-----------------|---------------|
| **Langchain** | Python | Множество моделей | Богатая экосистема | Средняя |
| **Qwen-Agent** | Python | Специализирован на Qwen | Интеграция с MCP | Низкая-средняя |
| **smolagents** | Python | Множество моделей | Стандартный набор | Низкая |


## Пример SE-агента 

### Инициализируем модель

In [1]:
import os

%load_ext dotenv
%dotenv .env

LLM_HOST = os.environ['LLM_HOST']
LLM_API_KEY = os.environ['LLM_API_KEY']
LLM_MODEL_NAME = os.environ['LLM_MODEL_NAME']

In [2]:
from langchain_openai import ChatOpenAI

model= ChatOpenAI(
    openai_api_base=LLM_HOST,
    openai_api_key=LLM_API_KEY,
    model_name=LLM_MODEL_NAME,
    temperature=0.1, # Низкая температура для детерминированного поведения
)

#response = model.invoke("Who are you?")
#print(response.content)

/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Определяем инструменты для Software Engineering

In [3]:
from langchain.tools import tool

@tool
def analyze_code_complexity(file_path: str) -> str:
    """
    Анализирует сложность кода в указанном файле.
    
    Args:
        file_path (str): Путь к файлу для анализа
        
    Returns:
        str: Отчет о сложности кода
    """
    try:
        with open(file_path, 'r') as file:
            code_content = file.read()
        
        # Простой анализ: подсчет строк и функций
        lines = code_content.split('\n')
        functions = [line for line in lines if line.strip().startswith('def ')]
        
        complexity_report = f"""
        Анализ файла: {file_path}
        - Общее количество строк: {len(lines)}
        - Количество функций: {len(functions)}
        - Сложность (оценка): {'Высокая' if len(functions) > 10 else 'Средняя' if len(functions) > 5 else 'Низкая'}
        """
        return complexity_report
    except Exception as e:
        return f"Ошибка при анализе файла: {str(e)}"

In [4]:
@tool
def run_unit_tests(test_path: str) -> str:
    """
    Запускает unit-тесты для проекта.
    
    Args:
        test_path (str): Путь к тестам или директории с тестами
        
    Returns:
        str: Результаты выполнения тестов
    """
    import subprocess
    
    try:
        result = subprocess.run(
            ['python', '-m', 'pytest', test_path, '-v'],
            capture_output=True, text=True, timeout=60
        )
        return f"Результаты тестов:\n{result.stdout}\nОшибки:\n{result.stderr}"
    except Exception as e:
        return f"Ошибка при запуске тестов: {str(e)}"

In [5]:
@tool
def search_documentation(query: str) -> str:
    """
    Ищет информацию в документации по программированию.
    
    Args:
        query (str): Поисковый запрос
        
    Returns:
        str: Релевантная информация из документации
    """
    # В реальной реализации здесь был бы вызов API документации
    documentation_sources = {
        "python": "https://docs.python.org/3/",
        "django": "https://docs.djangoproject.com/",
        "react": "https://reactjs.org/docs/"
    }
    
    return f"По запросу '{query}' проверьте документацию: {documentation_sources.get(query.lower(), 'https://docs.python.org/3/')}"

### Создаем агента с инструментами

In [6]:
from langchain.agents import create_agent

def create_se_agent():
    """
    Фабрика для создания агента Software Engineering.
    """
    # Собираем все инструменты
    tools = [analyze_code_complexity, run_unit_tests, search_documentation]
    
    # Создаем агента с системным промптом
    system_prompt = """
    Вы - AI ассистент для Software Engineering. Ваши задачи:
    1. Анализировать сложность кода и предлагать улучшения
    2. Запускать и анализировать unit-тесты
    3. Искать релевантную документацию
    4. Предлагать решения для проблем в коде
    
    Всегда запрашивайте уточнения, если задача нечеткая.
    Объясняйте свои действия и решения.
    """
    
    agent = create_agent(
        model=model,
        tools=tools,
        system_prompt=system_prompt
    )

    return agent

### Запуск агента

In [7]:
def example_agent_usage():
    """
    Демонстрация работы SE агента.
    """
    agent = create_se_agent()
    
    # Пример задачи для агента
    task = """
    Проанализируй сложность файла main.py, затем найди документацию по Python 
    по теме 'decorators' и предложи, как можно улучшить код если сложность высокая.
    """
    
    result = agent.invoke({
        "messages": [{"role": "user", "content": task}]
    })
    
    return result

In [8]:
snippet='''def my_decorator(func):
    def wrapper():
        print("Перед вызовом функции...")
        func()
        print("После вызова функции.")
    return wrapper

@my_decorator
def say_hello():
    print("Привет!")

# Вызов декорированной функции
say_hello()
'''

with open('main.py', 'w') as file:
    file.write(snippet)

In [9]:
example_result = example_agent_usage()

### Печать результата

In [10]:
def print_agent_messages_simple(result):
    """Компактное отображение сообщений агента"""
    
    if not isinstance(result, dict) or 'messages' not in result:
        print("Неверный формат результата")
        return
        
    for i, msg in enumerate(result['messages'], 1):
        msg_type = type(msg).__name__
        prefix = {
            'HumanMessage': 'Человек',
            'AIMessage': 'Ассистент', 
            'ToolMessage': 'Инструмент'
        }.get(msg_type, f'{msg_type}')
        
        content = getattr(msg, 'content', '').strip()
        
        print(f"\n{prefix} [{i}]:")
        if content:
            print(content)
        
        # Показываем вызовы инструментов
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            tools = [f"{tc.get('name', 'unknown')}({tc.get('args', {})})" 
                    for tc in msg.tool_calls]
            print(f"🔧 Вызывает: {', '.join(tools)}")

print_agent_messages_simple(example_result)


Человек [1]:
Проанализируй сложность файла main.py, затем найди документацию по Python 
    по теме 'decorators' и предложи, как можно улучшить код если сложность высокая.

Ассистент [2]:
Для выполнения запроса пользователя, я выполню следующие шаги:

1. **Анализ сложности кода** в файле `main.py`, используя инструмент `analyze_code_complexity`.
2. **Поиск документации** по теме "decorators" в Python с помощью инструмента `search_documentation`.
3. После получения результатов, я предложу улучшения кода, если будет выявлена высокая сложность.

Начну с первого шага — анализа сложности кода в файле `main.py`.
🔧 Вызывает: analyze_code_complexity({'file_path': 'main.py'})

Инструмент [3]:
Анализ файла: main.py
        - Общее количество строк: 14
        - Количество функций: 3
        - Сложность (оценка): Низкая

Ассистент [4]:
Сложность кода в файле `main.py` оказалась низкой, что говорит о простоте структуры и легкости поддержки. Тем не менее, я всё равно проведу поиск документации по 

# 3. MCP

**Model Context Protocol (MCP)** — это открытый стандарт, разработанный Anthropic, который обеспечивает унифицированную коммуникацию между большими языковыми моделями и внешними источниками данных и инструментами. MCP решает проблему интеграции $M \times N$, где ранее для каждой комбинации модели и инструмента требовалась своя интеграция.





## Ключевые компоненты MCP

| Компонент | Назначение | Пример в SE |
|-----------|------------|-------------|
| **Resources** | Объекты данных (документы, схемы БД) | Схемы баз данных, файлы конфигурации |
| **Tools** | Функции для выполнения действий | Запросы к БД, выполнение команд, вызовы API |
| **Prompts** | Шаблоны для эффективного взаимодействия | Промпты для рефакторинга, отладки, документирования |
| **Servers** | Программы, предоставляющие возможности | Серверы для Git, Docker, мониторинга кода |

##  Архитектура MCP

```
┌─────────────────┐    ┌──────────────────┐    ┌─────────────────┐
│   MCP-клиент    │    │   MCP Протокол   │    │  MCP-серверы    │
│                 │    │                  │    │                 │
│ - AI ассистенты │◄──►│ - JSON-RPC 2.0   │◄──►│ - GitHub        │
│ - IDE плагины   │    │ - Стандартизация │    │ - Docker        │
│ - Агенты        │    │ - Безопасность   │    │ - Базы данных   │
└─────────────────┘    └──────────────────┘    └─────────────────┘
```

## Преимущества и недостатки MCP

**Преимущества**:
- **Стандартизация**: Единый протокол вместо кастомных интеграций
- **Повторное использование**: Серверы MCP могут использоваться разными клиентами
- **Упрощение разработки**: Не нужно создавать интеграции для каждого источника данных
- **Модульность**: Легко добавлять новые инструменты без изменения основной системы

**Недостатки**:
- **Новая технология**: Экосистема развивается, ограниченная документация
- **Зависимость от стандарта**: Риск vendor lock-in если стандарт не получит широкого распространения
- **Производительность**: дополнительный уровень абстракции может влиять на скорость работы

## Сравнение: LangChain Agents vs MCP

| Критерий | LangChain Agents | MCP |
|----------|------------------|-----|
| **Интеграция инструментов** | Через кастомные Tools | Через стандартизированные серверы |
| **Переносимость** | Привязан к экосистеме LangChain | Независимый стандарт |
| **Сложность настройки** | Средняя | Низкая (благодаря стандартизации) |
| **Гибкость** | Высокая | Высокая |
| **Поддержка сообщества** | Большая | Растущая |


## Пример

Вот простой пример, демонстрирующий основные этапы работы MCP:

In [11]:
# server
import json

class MCPServer:
    def __init__(self):
        self.tools = {}
    
    def discovery(self):
        """Этап дискавери - возвращает список доступных инструментов"""
        return {
            "tools": list(self.tools.keys()),
            "protocol_version": "1.0"
        }
    
    def register_tool(self, name, function, description=""):
        """Регистрация нового инструмента"""
        self.tools[name] = {
            "function": function,
            "description": description
        }
    
    def call_tool(self, name, arguments):
        """Вызов инструмента с аргументами"""
        if name not in self.tools:
            return {"error": f"Tool {name} not found"}
        
        try:
            result = self.tools[name]["function"](**arguments)
            return {"content": result}
        except Exception as e:
            return {"error": str(e)}

In [12]:
# client
class MCPClient:
    def __init__(self, server):
        self.server = server
    
    def discover_tools(self):
        """Запрос списка доступных инструментов"""
        return self.server.discovery()
    
    def call_tool(self, name, **kwargs):
        """Вызов инструмента"""
        return self.server.call_tool(name, kwargs)

Пример использования

In [13]:
# Создаем сервер
server = MCPServer()

# Регистрируем простой инструмент
def add_numbers(a, b):
    """Складывает два числа"""
    return a + b

server.register_tool("add", add_numbers, "Сложение двух чисел")


# Регистрируем второй инструмент с проверкой ошибок
def divide_numbers(a, b):
    if b == 0:
        raise ValueError("Division by zero")
    return a / b

server.register_tool("divide", divide_numbers, "Деление двух чисел")

In [14]:
# Создаем клиент
client = MCPClient(server)

In [15]:
# Дискавери
tools = client.discover_tools()
print("Доступные инструменты:", tools)

# Вызов инструмента
result = client.call_tool("add", a=5, b=3)
print("Результат сложения:", result)

# Тестирование обработки ошибок
result = client.call_tool("divide", a=10, b=0)
print("Результат деления:", result)

Доступные инструменты: {'tools': ['add', 'divide'], 'protocol_version': '1.0'}
Результат сложения: {'content': 8}
Результат деления: {'error': 'Division by zero'}


Этот пример показывает:

1. **Дискавери** - клиент запрашивает список доступных инструментов
2. **Регистрацию инструмента** - сервер регистрирует функцию сложения
3. **Вызов инструмента** - клиент использует зарегистрированный инструмент

Он демонстрирует:
- Структуру MCP-взаимодействия
- Динамическую регистрацию инструментов
- Простой JSON-формат сообщений
- Обработку результатов и ошибок

Пример максимально упрощен, но показывает основную логику работы протокола MCP.
В реальном MCP протокол сложнее и включает:
- JSON-RPC сообщения
- Ресурсы и инструменты
- Стандартизированные форматы сообщений
- Сетевую коммуникацию

Чуть более сложный пример

In [16]:
import json

# Сервер MCP - предоставляет инструменты, НЕ содержит LLM
class MCPServer:
    def __init__(self):
        self.tools = {}
    
    def discovery(self):
        """Сервер сообщает клиенту, какие инструменты доступны"""
        return {
            "tools": [
                {
                    "name": name,
                    "description": tool["description"],
                    "parameters": tool["parameters"]
                }
                for name, tool in self.tools.items()
            ]
        }
    
    def register_tool(self, name, function, description, parameters):
        """Регистрация инструмента с описанием для LLM"""
        self.tools[name] = {
            "function": function,
            "description": description,
            "parameters": parameters
        }
    
    def call_tool(self, name, arguments):
        """Сервер выполняет только инструмент, НЕ LLM"""
        if name not in self.tools:
            return {"error": f"Tool {name} not found"}
        
        try:
            result = self.tools[name]["function"](**arguments)
            return {"content": [{"type": "text", "text": str(result)}]}
        except Exception as e:
            return {"error": str(e)}

# Клиент MCP - содержит LLM, использует инструменты сервера
class MCPClient:
    def __init__(self, server):
        self.server = server
        # Клиент имеет доступ к LLM (в реальности это Claude, GPT и т.д.)
        # В этом примере мы имитируем работу LLM
    
    def process_user_query(self, user_input):
        """Клиент обрабатывает запрос пользователя с помощью LLM"""
        print(f"Пользователь: {user_input}")
        
        # 1. Дискавери - LLM узнает какие инструменты доступны
        available_tools = self.server.discovery()
        print(f"Доступные инструменты: {[tool['name'] for tool in available_tools['tools']]}")
        
        # 2. LLM решает, какой инструмент использовать
        tool_to_use, tool_args = self._llm_decide_tool(user_input, available_tools)
        
        if tool_to_use:
            # 3. Клиент вызывает инструмент на сервере
            print(f"LLM решил использовать инструмент: {tool_to_use} с аргументами: {tool_args}")
            result = self.server.call_tool(tool_to_use, tool_args)
            
            # 4. LLM обрабатывает результат и формирует ответ
            final_response = self._llm_generate_response(user_input, result)
            return final_response
        else:
            return "LLM решил, что для этого запроса не нужны инструменты"

    def _llm_decide_tool(self, user_input, available_tools):
        """Имитация LLM, решающей какой инструмент использовать"""
        # В реальности здесь сложная логика LLM
        # В нашем примере - простые правила
        
        user_input_lower = user_input.lower()
        
        if "погода" in user_input_lower:
            return "get_weather", {"location": "Москва"}  # упрощенно
        elif "переведи" in user_input_lower:
            # Извлекаем текст для перевода (упрощенно)
            text_to_translate = user_input.replace("переведи", "").strip()
            return "translate", {"text": text_to_translate, "target_lang": "en"}
        elif "посчитай" in user_input_lower:
            # Извлекаем числа (упрощенно)
            numbers = [int(s) for s in user_input.split() if s.isdigit()]
            if len(numbers) >= 2:
                return "calculate", {"operation": "add", "a": numbers[0], "b": numbers[1]}
        
        return None, None

    def _llm_generate_response(self, user_input, tool_result):
        """Имитация LLM, формирующей финальный ответ на основе результата инструмента"""
        if "error" in tool_result:
            return f"Произошла ошибка: {tool_result['error']}"
        
        result_text = tool_result['content'][0]['text']
        return f"На основе данных инструмента: {result_text}"

# Пример использования
if __name__ == "__main__":
    # Создаем сервер
    server = MCPServer()
    
    # Регистрируем инструменты с описаниями для LLM
    def get_weather(location):
        """Получить погоду для локации"""
        # В реальности здесь API запрос к погодному сервису
        return f"Погода в {location}: +20°C, солнечно"
    
    def translate(text, target_lang):
        """Перевести текст на целевой язык"""
        # В реальности здесь вызов переводческого API
        return f"Перевод '{text}' на {target_lang}: translated_text"
    
    def calculate(operation, a, b):
        """Выполнить математическую операцию"""
        if operation == "add":
            return a + b
        elif operation == "multiply":
            return a * b
    
    server.register_tool(
        "get_weather",
        get_weather,
        "Получить текущую погоду для указанной локации",
        {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "Город или локация"}
            }
        }
    )
    
    server.register_tool(
        "translate", 
        translate,
        "Перевести текст на другой язык",
        {
            "type": "object", 
            "properties": {
                "text": {"type": "string", "description": "Текст для перевода"},
                "target_lang": {"type": "string", "description": "Целевой язык"}
            }
        }
    )
    
    server.register_tool(
        "calculate",
        calculate,
        "Выполнить математическую операцию",
        {
            "type": "object",
            "properties": {
                "operation": {"type": "string", "enum": ["add", "multiply"]},
                "a": {"type": "number"},
                "b": {"type": "number"}
            }
        }
    )
    
    # Создаем клиента (который содержит LLM)
    client = MCPClient(server)
    
    # Пользователь взаимодействует с клиентом (который использует LLM + инструменты)
    queries = [
        "Какая погода в Москве?",
        "Переведи 'привет мир' на английский",
        "Посчитай 5 + 3",
        "Расскажи анекдот"  # Для этого запроса не нужны инструменты
    ]
    
    for query in queries:
        print("\n" + "="*50)
        response = client.process_user_query(query)
        print(f"Ответ: {response}")


Пользователь: Какая погода в Москве?
Доступные инструменты: ['get_weather', 'translate', 'calculate']
LLM решил использовать инструмент: get_weather с аргументами: {'location': 'Москва'}
Ответ: На основе данных инструмента: Погода в Москва: +20°C, солнечно

Пользователь: Переведи 'привет мир' на английский
Доступные инструменты: ['get_weather', 'translate', 'calculate']
LLM решил использовать инструмент: translate с аргументами: {'text': "Переведи 'привет мир' на английский", 'target_lang': 'en'}
Ответ: На основе данных инструмента: Перевод 'Переведи 'привет мир' на английский' на en: translated_text

Пользователь: Посчитай 5 + 3
Доступные инструменты: ['get_weather', 'translate', 'calculate']
LLM решил использовать инструмент: calculate с аргументами: {'operation': 'add', 'a': 5, 'b': 3}
Ответ: На основе данных инструмента: 8

Пользователь: Расскажи анекдот
Доступные инструменты: ['get_weather', 'translate', 'calculate']
Ответ: LLM решил, что для этого запроса не нужны инструменты


# Упражнение

Создайте агента на основе Qwen3-0.6B или другой выбранной модели, который по заданию и по тестам (примеры входа и выхода) генерирует код, запускает его на тестах, и на основе ошибок поправляет его.

# Ссылки

- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents) 
- [Model Context Protocol](https://github.com/modelcontextprotocol) 